# Imports

In [1]:
import numpy as np
from scipy.integrate import solve_ivp

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import linregress
import matplotlib.cm as cm
from pathlib import Path

# Units

In [2]:
G_real = 6.67430e-11  # m^3 kg^-1 s^-2
M_ref = 1.989e30  # Mass of Sun (kg)
L_ref = 1.496e11  # 1 AU (meters)

# T = sqrt( L^3 / (G * M) )
T_ref = np.sqrt(L_ref**3 / (G_real * M_ref))
V_ref = L_ref / T_ref

print(f"1 Mass Unit      = {M_ref:.2e} kg")
print(f"1 Distance Unit  = {L_ref:.2e} m")
print(f"1 Time Unit      = {T_ref:.2e} seconds ({T_ref / 86400:.2f} days)")
print(f"1 Velocity Unit  = {V_ref:.2f} m/s")

1 Mass Unit      = 1.99e+30 kg
1 Distance Unit  = 1.50e+11 m
1 Time Unit      = 5.02e+06 seconds (58.13 days)
1 Velocity Unit  = 29788.90 m/s


# Solver

In [3]:
class System:
    """
    Class simulating the 3-Body Problem using numerical integration.
    """

    def __init__(
        self,
        masses: list[float],
        initial_positions: np.ndarray,
        initial_velocities: np.ndarray,
        time: float,
        time_steps: int,
        names: list[str] | None = None,
        colors: list[str] | None = None,
        scenario: str | None = None,
    ):
        """
        Initialize the system parameters and state vectors.
        """
        self.masses = masses
        self.initial_positions = initial_positions
        self.initial_velocities = initial_velocities
        self.scenario = scenario

        # Flatten initial positions and velocities into a single 1D state vector (size 18)
        # Structure: [x1, y1, z1, x2, ..., vx1, vy1, vz1, vx2, ...]
        self.inital_state = np.concatenate(
            (initial_positions.flatten(), initial_velocities.flatten())
        )
        self.time = time
        self.time_steps = time_steps
        self.time_span = (0, time)  # Time span in normalized units
        self.t_eval = np.linspace(0, time, time_steps)

        # Set default names
        if names is None:
            self.names = [f"Body {i + 1}" for i in range(len(masses))]
        else:
            self.names = names

        # Set default "Neon" colors
        if colors is None:
            self.colors = ["#FFFFFF", "#75F9FD", "#FD51E6"]
        else:
            self.colors = colors

    def three_body_equations(self, t, state, masses):
        """
        Computes the derivatives of the system state (velocities and accelerations).
        Used by the ODE solver.
        """
        # The 'state' comes in as a 1D array of 18 numbers.
        # It needs to be reshaped into two 3x3 matrices for vector calculations.

        # First 9 elements: Positions (r)
        r = state[:9].reshape((3, 3))

        # Last 9 elements: Velocities (v)
        v = state[9:].reshape((3, 3))

        # Derivative of position is velocity (dr/dt = v)
        drdt = v
        dvdt = np.zeros((3, 3))
        G = 1.0  # Normalized gravitational constant

        # Calculate gravitational acceleration for each body
        for i in range(3):
            for j in range(3):
                if i != j:
                    # Calculate distance vector components
                    dx = r[j, 0] - r[i, 0]  # x_j - x_i
                    dy = r[j, 1] - r[i, 1]  # y_j - y_i
                    dz = r[j, 2] - r[i, 2]  # z_j - z_i

                    # Euclidean distance
                    dist = np.sqrt(dx**2 + dy**2 + dz**2)

                    # Newton's Law of Universal Gravitation: F = G * m1 * m2 / r^3 * vec(r)
                    # We calculate acceleration directly (F/m1), so we multiply by masses[j]
                    factor = 1 * G * masses[j] / (dist**3)

                    dvdt[i, 0] += factor * dx  # Acceleration in X
                    dvdt[i, 1] += factor * dy  # Acceleration in Y
                    dvdt[i, 2] += factor * dz  # Acceleration in Z

        # The solver expects a single flat list of 18 derivatives.
        # Format: [velocities..., accelerations...]
        return np.concatenate((drdt.flatten(), dvdt.flatten()))

    def run_simulation(self):
        """
        Executes the numerical integration using Scipy's solve_ivp.
        """
        solution = solve_ivp(
            self.three_body_equations,
            self.time_span,
            self.inital_state,
            args=(self.masses,),
            method="DOP853",  # High-order Runge-Kutta method
            t_eval=self.t_eval,
            rtol=1e-6,
            atol=1e-9,
        )

        # Reshape results into 3D array: (3 bodies, 3 coordinates, N time steps)
        self.r_sol = solution.y[:9].reshape((3, 3, -1))

    def create_gif(self, filename: str):
        """
        Generates and saves an animated GIF of the simulation with visual effects.
        """
        # Run simulation if results are not available
        if not hasattr(self, "r_sol"):
            self.run_simulation()

        r_sol = self.r_sol
        steps = r_sol.shape[2]

        plt.style.use("dark_background")
        fig_color = "black"

        # Create figure with extra width for HUD text (12x8 inches)
        fig = plt.figure(figsize=(12, 8), facecolor=fig_color)
        ax = fig.add_subplot(111, projection="3d")
        ax.set_facecolor(fig_color)

        # Configure subtle grid lines
        ax.grid(color="white", linestyle=":", linewidth=0.3, alpha=0.2)

        # Remove default gray background panels of the 3D plot
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False

        info_text = "SYSTEM PARAMETERS (t=0):\n"
        info_text += "=" * 25 + "\n\n"

        for i, name in enumerate(self.names):
            m = self.masses[i]
            r = self.initial_positions[i]
            v = self.initial_velocities[i]

            # Calculate initial distance from center and total velocity
            dist = np.linalg.norm(r)
            vel_mag = np.linalg.norm(v)

            info_text += f"[{name.upper()}]\n"
            info_text += f" Mass: {m:.2e}\n"
            info_text += f" R_start: {dist:.2f} AU\n"
            info_text += f" V_start: {vel_mag:.2f}\n"

            # Display exact vector components
            info_text += f" Pos: [{r[0]:.2f}, {r[1]:.2f}, {r[2]:.2f}]\n"
            info_text += f" Vel: [{v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f}]\n"
            info_text += "-" * 25 + "\n"

        fig.text(
            0.02,
            0.90,
            info_text,
            fontsize=9,
            fontfamily="monospace",
            color="white",
            verticalalignment="top",
            horizontalalignment="left",
        )

        # Each trajectory consists of three overlapping lines (layers) for a neon effect.
        # Layer config: [(Line Width, Alpha), ...]
        glow_lines = []
        glow_layers_config = [(1.5, 1.0), (4, 0.4), (8, 0.1)]

        for i in range(3):
            body_stack = []
            base_color = self.colors[i]
            # Create 3 lines per body
            for lw, alpha in glow_layers_config:
                (line,) = ax.plot(
                    [],
                    [],
                    [],
                    color=base_color,
                    linewidth=lw,
                    alpha=alpha,
                    solid_capstyle="round",
                    zorder=2,
                )
                body_stack.append(line)
            glow_lines.append(body_stack)

        # Initialize glowing spheres for bodies (Outer Glow + Inner Core)
        points_glow = [
            ax.plot(
                [],
                [],
                [],
                marker="o",
                color=self.colors[i],
                markersize=12,
                alpha=0.3,
                zorder=3,
            )[0]
            for i in range(3)
        ]
        points_core = [
            ax.plot(
                [],
                [],
                [],
                marker="o",
                color=self.colors[i],
                markersize=6,
                alpha=1.0,
                zorder=4,
            )[0]
            for i in range(3)
        ]

        # Calculate global min/max to ensure a 1:1:1 aspect ratio.

        all_x = r_sol[:, 0, :].flatten()
        all_y = r_sol[:, 1, :].flatten()
        all_z = r_sol[:, 2, :].flatten()

        # Find the maximum span across any dimension
        max_range = (
            np.array(
                [
                    all_x.max() - all_x.min(),
                    all_y.max() - all_y.min(),
                    all_z.max() - all_z.min(),
                ]
            ).max()
            / 2.0  # Convert diameter to radius
        )

        # Calculate the geometric center
        mid_x = (all_x.max() + all_x.min()) * 0.5
        mid_y = (all_y.max() + all_y.min()) * 0.5
        mid_z = (all_z.max() + all_z.min()) * 0.5

        # Set symmetric limits around the center
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)

        ax.set_xlabel("X (AU)")
        ax.set_ylabel("Y (AU)")
        ax.set_zlabel("Z (AU)")
        ax.set_title(f"3-Body Problem Simulation \n {self.scenario}")

        ax.legend(
            [p for p in points_core],
            self.names,
            loc="upper right",
            facecolor="black",
            edgecolor="gray",
        )

        def update(frame):
            artists_to_return = []  # Optimization: return only modified artists

            # Define a sliding window [start : frame]
            tail_len = 100
            start_trace = max(0, frame - tail_len)

            for i in range(3):
                # 1. Slice trajectory data for the trail
                x_trail = r_sol[i, 0, start_trace:frame]
                y_trail = r_sol[i, 1, start_trace:frame]
                z_trail = r_sol[i, 2, start_trace:frame]

                # 2. Get current position (head)
                x_point = [r_sol[i, 0, frame]]
                y_point = [r_sol[i, 1, frame]]
                z_point = [r_sol[i, 2, frame]]

                # 3. Update all 3 layers of the neon line
                for layer_line in glow_lines[i]:
                    layer_line.set_data(x_trail, y_trail)
                    layer_line.set_3d_properties(z_trail)
                    artists_to_return.append(layer_line)

                # 4. Update the body spheres (Glow + Core)
                points_glow[i].set_data(x_point, y_point)
                points_glow[i].set_3d_properties(z_point)

                points_core[i].set_data(x_point, y_point)
                points_core[i].set_3d_properties(z_point)

                artists_to_return.append(points_glow[i])
                artists_to_return.append(points_core[i])

            # Slowly rotate camera for 3D effect
            ax.view_init(elev=20, azim=frame * 0.3)

            return artists_to_return

        print(f"Generating animation ({steps} frames)...")

        ani = FuncAnimation(fig, update, frames=steps, interval=20, blit=False)

        # Save using Pillow (GIF writer)
        ani.save(
            filename, writer="pillow", fps=30, savefig_kwargs={"facecolor": "black"}
        )
        print(f"Done! Saved as: {filename}")
        plt.close()

    def save_neon_pdf(self, folder_name: str, filename: str):
        """
        Save static .pdf
        """

        # 1. Simualtion
        if not hasattr(self, "r_sol"):
            self.run_simulation()

        # 2. Folder and path
        output_dir = Path(folder_name)
        full_path = output_dir / filename

        # 3. (Style (coppied form create_gif))
        plt.style.use("dark_background")
        fig_color = "black"

        fig = plt.figure(figsize=(12, 8), facecolor=fig_color)
        ax = fig.add_subplot(111, projection="3d")
        ax.set_facecolor(fig_color)

        ax.grid(color="white", linestyle=":", linewidth=0.3, alpha=0.2)
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False

        # 4. HUD
        info_text = "SYSTEM PARAMETERS (t=0):\n"
        info_text += "=" * 25 + "\n\n"

        for i, name in enumerate(self.names):
            m = self.masses[i]
            r = self.initial_positions[i]
            v = self.initial_velocities[i]

            # Calculate initial distance from center and total velocity
            dist = np.linalg.norm(r)
            vel_mag = np.linalg.norm(v)

            info_text += f"[{name.upper()}]\n"
            info_text += f" Mass: {m:.2e}\n"
            info_text += f" R_start: {dist:.2f} AU\n"
            info_text += f" V_start: {vel_mag:.2f}\n"

            # Display exact vector components
            info_text += f" Pos: [{r[0]:.2f}, {r[1]:.2f}, {r[2]:.2f}]\n"
            info_text += f" Vel: [{v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f}]\n"
            info_text += "-" * 25 + "\n"

        fig.text(
            0.02,
            0.90,
            info_text,
            fontsize=9,
            fontfamily="monospace",
            color="white",
            verticalalignment="top",
            horizontalalignment="left",
        )

        glow_layers_config = [(1.5, 1.0), (4, 0.4), (8, 0.1)]
        points_core = []

        for i in range(3):
            xs = self.r_sol[i, 0, :]
            ys = self.r_sol[i, 1, :]
            zs = self.r_sol[i, 2, :]
            base_color = self.colors[i]

            for lw, alpha in glow_layers_config:
                ax.plot(
                    xs,
                    ys,
                    zs,
                    color=base_color,
                    linewidth=lw,
                    alpha=alpha,
                    solid_capstyle="round",
                    zorder=2,
                )

            ax.plot(
                [xs[-1]],
                [ys[-1]],
                [zs[-1]],
                marker="o",
                color=base_color,
                markersize=12,
                alpha=0.3,
                zorder=3,
            )

            core_plot = ax.plot(
                [xs[-1]],
                [ys[-1]],
                [zs[-1]],
                marker="o",
                color=base_color,
                markersize=6,
                alpha=1.0,
                zorder=4,
            )[0]

            points_core.append(core_plot)

        all_x = self.r_sol[:, 0, :].flatten()
        all_y = self.r_sol[:, 1, :].flatten()
        all_z = self.r_sol[:, 2, :].flatten()
        max_range = (
            np.array(
                [
                    all_x.max() - all_x.min(),
                    all_y.max() - all_y.min(),
                    all_z.max() - all_z.min(),
                ]
            ).max()
            / 2.0
        )
        mid_x, mid_y, mid_z = (
            (all_x.max() + all_x.min()) * 0.5,
            (all_y.max() + all_y.min()) * 0.5,
            (all_z.max() + all_z.min()) * 0.5,
        )

        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)

        ax.set_xlabel("X (AU)")
        ax.set_ylabel("Y (AU)")
        ax.set_zlabel("Z (AU)")
        ax.set_title(f"3-Body Problem Simulation (Full trace) \n {self.scenario}")

        ax.legend(
            points_core,
            self.names,
            loc="upper right",
            facecolor="black",
            edgecolor="gray",
            labelcolor="white",
        )

        ax.view_init(elev=20, azim=45)

        plt.savefig(
            full_path,
            format="pdf",
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
            edgecolor="none",
        )
        print(f"Saved: {full_path}")
        plt.close()

# Functions

In [4]:
def analyze_lapunov(
    system_class,
    scenario_name,
    file_name,
    masses,
    r_init,
    v_init,
    time_end,
    names: None = None,
):
    print(f"--- Full analysis (9 cases): {scenario_name} ---")

    # Dict for results
    results = {}

    # Reference simulation
    sim_ref = system_class(masses, r_init, v_init, time=time_end, time_steps=2000)
    sim_ref.run_simulation()

    results["Reference"] = sim_ref.r_sol
    t = sim_ref.t_eval

    # Plotting
    plt.figure(figsize=(14, 8), facecolor="white")
    plt.style.use("default")

    axis_names = ["X", "Y", "Z"]
    if names is not None:
        body_names = names
    else:
        body_names = ["Body 1", "Body 2", "Body 3"]
    epsilon = 1e-12

    case_counter = 0
    colors = cm.turbo(np.linspace(0, 1, 9))

    for body_idx in range(3):
        for axis_idx in range(3):
            # Dictioanry key for the case
            label_text = f"{body_names[body_idx]} - axis {axis_names[axis_idx]}"

            # Perturbation
            r_pert = r_init.copy()
            r_pert[body_idx, axis_idx] += epsilon

            sim_pert = system_class(
                masses, r_pert, v_init, time=time_end, time_steps=2000
            )
            sim_pert.run_simulation()

            results[label_text] = sim_pert.r_sol

            # log(diff)
            diff = results["Reference"] - results[label_text]
            dists = np.linalg.norm(diff, axis=(0, 1))
            log_dists = np.log(dists + 1e-16)

            line_styles = ["-", "--", ":"]
            plt.plot(
                t / (2 * np.pi),
                log_dists,
                label=label_text,
                color=colors[case_counter],
                linestyle=line_styles[body_idx],
                linewidth=1.5,
                alpha=0.8,
            )

            case_counter += 1

    plt.title(f"Sensitivity analysis: {scenario_name}")
    plt.xlabel("Time")
    plt.ylabel(r"$\ln(\delta(t))$")
    plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(file_name)

    return results, t

In [5]:
def analyze_lyapunov_per_body(
    system_class,
    scenario_name,
    file_name,
    masses,
    r_init,
    v_init,
    time_end,
    pert_body_idx,
    pert_axis_idx,
    names: None = None,
    time_steps=5000,
):
    """
    Analyzes the error evolution and the evolution of the Lyapunov exponent over time.
    pert_body_idx: Index of the body being perturbed (0, 1, or 2)
    pert_axis_idx: Axis of perturbation (0=X, 1=Y, 2=Z)
    """
    if names is not None:
        body_names = names
    else:
        body_names = ["Body 1", "Body 2", "Body 3"]
    axis_names = ["X", "Y", "Z"]

    # 1. Reference Simulation
    sim_ref = system_class(masses, r_init, v_init, time=time_end, time_steps=time_steps)
    sim_ref.run_simulation()

    # 2. Perturbed Simulation
    epsilon = 1e-12
    r_pert = r_init.copy()
    r_pert[pert_body_idx, pert_axis_idx] += epsilon  # Perturb only one body

    sim_pert = system_class(
        masses, r_pert, v_init, time=time_end, time_steps=time_steps
    )
    sim_pert.run_simulation()

    # 3. Calculations
    diff = sim_ref.r_sol - sim_pert.r_sol
    t = sim_ref.t_eval

    # Create two subplots, one below the other
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12), sharex=True)
    plt.style.use("default")

    colors = ["gold", "dodgerblue", "red"]

    for i in range(3):
        # A. Calculate Euclidean distance in 3D for body 'i'
        dist_body = np.linalg.norm(diff[i], axis=0)

        # Avoid log(0) by adding a small constant where dist=0
        dist_body = np.maximum(dist_body, 1e-20)

        # Logarithm of distance (Separation)
        log_dist = np.log(dist_body)

        # Line style
        linestyle = "-" if i == pert_body_idx else "--"
        linewidth = 2.5 if i == pert_body_idx else 1.5
        alpha = 1.0 if i == pert_body_idx else 0.7

        # Linear regression
        idx_start = int(len(t) * 0.05)  # Skip the first 10%
        idx_stop = int(len(t) * 0.2)
        res = linregress(t[idx_start:idx_stop], log_dist[idx_start:idx_stop])

        ax1.plot(
            t / (2 * np.pi),  # Time in periods (optional)
            log_dist,
            label=rf"{body_names[i]}: $\lambda \approx {res.slope:.4f}$",
            color=colors[i],
            linestyle=linestyle,
            linewidth=linewidth,
            alpha=alpha,
        )

        # Formula: lambda(t) = 1/t * ln( d(t) / d(0) )

        t_safe = t[1:]
        dist_safe = dist_body[1:]

        # Divide by epsilon, as this was the initial input perturbation
        ftle_series = (1 / t_safe) * np.log(dist_safe / epsilon)

        ax2.plot(
            t_safe / (2 * np.pi),
            ftle_series,
            label=f"{body_names[i]}",
            color=colors[i],
            linestyle=linestyle,
            linewidth=linewidth,
            alpha=alpha,
        )

    ax1.set_title(
        f"{scenario_name} \n Perturbed: {body_names[pert_body_idx]} (Axis {axis_names[pert_axis_idx]})"
    )
    ax1.set_ylabel(r"$\ln(\delta(t))$")
    ax1.grid(True, which="both", linestyle=":", alpha=0.6)
    ax1.legend(loc="upper left")

    # Aesthetics for Plot 2 (Lyapunov Exponent)
    ax2.set_title(r"Evolution of Lyapunov Exponent $\lambda(t)$")
    ax2.set_xlabel("Time")
    ax2.set_ylabel(r"$\lambda(t) = \frac{1}{t} \ln(\frac{\delta(t)}{\epsilon})$")
    ax2.grid(True, which="both", linestyle=":", alpha=0.6)

    # Reference line at zero (stability boundary)
    ax2.axhline(0, color="black", linewidth=1, linestyle="-")
    ax2.set_ylim(-1.5, 1.5)

    plt.tight_layout()
    plt.savefig(file_name)

# Simulation

In [6]:
# MASSES (Relative to the Sun)
# ------------------------------------------
m_sun = 1.0  # Sun mass (Reference unit)
m_earth = 3.003e-6
m_mars = 3.213e-7
m_jupiter = 9.543e-4
m_moon = 3.694e-8

# DISTANCES (Average Semi-major axis in AU)
# ------------------------------------------
r_earth = 1.000  # 1 AU (Distance Sun-Earth)
r_mars = 1.524
r_jupiter = 5.203

# ORBITAL VELOCITIES (Normalized)
# Circular orbit velocity formula: v = sqrt(GM / r).
# Since G=1 and M_Sun=1, this simplifies to: v = sqrt(1 / r)
# ------------------------------------------
v_earth = 1.0
v_mars = np.sqrt(1.0 / r_mars)
v_jupiter = np.sqrt(1.0 / r_jupiter)

## Sun, Earth, Mars

### Initial parameters

In [40]:
masses_SuEaMa = [m_sun, m_earth, m_mars]

r_initial_SuEaMa = np.array(
    [
        [0.0, 0.0, 10e-2],
        [r_earth, 0.0, 0.0],
        [-r_mars, 0.0, 0.0],
    ]
)

v_drift = 0.0
v_initial_SuEaMa = np.array(
    [
        [0.0, 0.0, v_drift],
        [0.0, v_earth, v_drift],
        [0.0, -v_mars, v_drift],
    ]
)

time_SuEaMa = 10.0 * 2 * np.pi

simulation_sueama = System(
    masses_SuEaMa,
    r_initial_SuEaMa,
    v_initial_SuEaMa,
    time_SuEaMa,
    500,
    names=["Sun", "Earth", "Mars"],
    scenario="Sun_Earth_Mars",
)

SuEaMa_folder = Path("SunEarthMars")

### GIF

In [41]:
simulation_sueama.create_gif(SuEaMa_folder / "sun_earth_mars_pert.gif")

Generating animation (500 frames)...
Done! Saved as: SunEarthMars/sun_earth_mars_pert.gif


### Full trace

In [ ]:
simulation_sueama.save_neon_pdf("SunEarthMars", "SuEaMa_trajektoria_3d.pdf")

### 9D Lapunow

In [ ]:
r_init_pert_SuEaMa = analyze_lapunov(
    System,
    "Sun_Earth_Mars",
    SuEaMa_folder / "all_lapunovs_Su_Ea_Ma.pdf",
    masses_SuEaMa,
    r_initial_SuEaMa,
    v_initial_SuEaMa,
    time_end=time_SuEaMa,
    names=["Sun", "Earth", "Mars"],
)

### 3D Lapunow

In [ ]:
for body in range(3):
    for axis in range(3):
        analyze_lyapunov_per_body(
            System,
            "Sun_Earth_Mars",
            SuEaMa_folder / f"Su_Ea_Ma_lapunov_{body}_{axis}.pdf",
            masses_SuEaMa,
            r_initial_SuEaMa,
            v_initial_SuEaMa,
            time_end=time_SuEaMa,
            pert_body_idx=body,
            pert_axis_idx=axis,
            names=["Sun", "Earth", "Mars"],
        )

## Figure-8 

### Initial parameters

In [ ]:
masses_figure8 = [m_sun, m_sun, m_sun]

r_initial_figure8 = np.array(
    [[0.97000436, -0.24308753, 0], [-0.97000436, 0.24308753, 0], [0, 0, 0]]
)

v3 = np.array([-0.93240737, -0.86473146, 0])
v_initial_figure8 = np.array([-v3 / 2, -v3 / 2, v3])


time_figure8 = 15 * 2 * np.pi

simulation_f8 = System(
    masses_figure8,
    r_initial_figure8,
    v_initial_figure8,
    time_figure8,
    500,
    scenario="Figure - 8",
)

figure8_folder = Path("Figure8")

### GIF

In [ ]:
simulation_f8.create_gif(figure8_folder / "figure8.gif")

Generating animation (500 frames)...
Done! Saved as: Figure8/figure8_pert.gif


### Full trace

In [ ]:
simulation_f8.save_neon_pdf("Figure8", "f8_trajektoria_3d.pdf")

### 9D Lapunow

In [ ]:
r_init_pert_figure8 = analyze_lapunov(
    System,
    "Figure-8",
    figure8_folder / "all_lapunovs_figure8.pdf",
    masses_figure8,
    r_initial_figure8,
    v_initial_figure8,
    time_end=time_figure8,
)

### 3D Lapunow

In [ ]:
for body in range(3):
    for axis in range(3):
        analyze_lyapunov_per_body(
            System,
            "Figure-8",
            figure8_folder / f"figure8_lapunov_{body}_{axis}.pdf",
            masses_figure8,
            r_initial_figure8,
            v_initial_figure8,
            time_end=time_figure8,
            pert_body_idx=body,
            pert_axis_idx=axis,
        )

## Pyhagorean 3 body problem

### Initial parameters

In [ ]:
masses_pythagorean = [3.0, 4.0, 5.0]

r_initial_pythagorean = np.array([[1.0, 3.0, 0], [-2.0, -1.0, 0], [1.0, -1.0, 0]])

v_initial_pythagorean = np.zeros((3, 3))

time_pythagorean = 10.0 * 2 * np.pi

simulation_pyth = System(
    masses_pythagorean,
    r_initial_pythagorean,
    v_initial_pythagorean,
    time_pythagorean,
    500,
    scenario="Pythagoras",
)
pitagoras_folder = Path("Pitagoras")

### GIF

In [ ]:
simulation_pyth.create_gif(pitagoras_folder / "Pythagoras.gif")

Generating animation (500 frames)...
Done! Saved as: Pitagoras/Pythagoras_pert.gif


### Full trace

In [ ]:
simulation_pyth.save_neon_pdf("Pitagoras", "pyth_trajektoria_3d.pdf")

### 9D Lapunow

In [ ]:
r_init_pert_pythagorean = analyze_lapunov(
    System,
    "Pythagoras",
    pitagoras_folder / "all_lapunovs_pitagoras.pdf",
    masses_pythagorean,
    r_initial_pythagorean,
    v_initial_pythagorean,
    time_end=time_pythagorean,
)

### 3D Lapunow

In [ ]:
for body in range(3):
    for axis in range(3):
        analyze_lyapunov_per_body(
            System,
            "Pythagoras",
            pitagoras_folder / f"pitagoras_lapunov_{body}_{axis}.pdf",
            masses_pythagorean,
            r_initial_pythagorean,
            v_initial_pythagorean,
            time_end=time_pythagorean,
            pert_body_idx=body,
            pert_axis_idx=axis,
        )